# OpenFlights Analytics
This notebook demonstrates data engineering and analytics using DuckDB and dbt with the OpenFlights dataset.


The first section validates the ingestion layer and compares raw row counts across the primary OpenFlights source tables: airports, airlines, and routes. This helps confirm the dataset shape before exploring curated dbt mart results.

In [2]:
import duckdb
import pandas as pd
from pathlib import Path

repo_root = Path('..')
db_path = repo_root / 'data' / 'analytics.duckdb'
con = duckdb.connect(str(db_path))

# Check raw row counts
counts = con.execute('''
select
  'airports' as dataset,
  count(*) as rows
from raw.airports
union all
select
  'airlines',
  count(*)
from raw.airlines
union all
select
  'routes',
  count(*)
from raw.routes
''').df()
counts


,dataset,rows
0,airports,7698
1,airlines,6162
2,routes,67663


These visualizations draw from the dbt mart models and highlight the strongest network patterns in the OpenFlights dataset.
- The first chart ranks the top countries by airport count, showing where airport infrastructure is most concentrated.
- The second chart ranks airlines by route count, showing which carriers have the most extensive route networks.
- The third chart ranks airports by route activity, identifying the busiest hubs that anchor flight connectivity.

Each chart is intended to surface a different aspect of the aviation network: country coverage, airline reach, and hub intensity.


In [ ]:
# Explore analytic model outputs
airport_counts = con.execute('select * from main.mart_top_countries_by_airports').df()
route_counts = con.execute('select * from main.mart_top_airlines_by_routes').df()
hub_airports = con.execute('select * from main.mart_top_hub_airports').df()

print('Top countries by airport count')
print(airport_counts.head(10))
print('\nTop airlines by route count')
print(route_counts.head(10))
print('\nTop hub airports by total route activity')
print(hub_airports.head(10))

import plotly.express as px

# Country coverage: which nations have the largest airport networks?
fig = px.bar(airport_counts.head(10), x='country', y='airport_count', title='Top 10 Countries by Airport Count')
fig.show()

# Airline reach: which carriers operate the most routes in this dataset?
fig = px.bar(route_counts.head(10), x='airline', y='route_count', title='Top 10 Airlines by Route Count')
fig.show()

# Hub activity: the top airports by number of routes served.
fig = px.bar(hub_airports.head(10), x='name', y='total_routes', title='Top 10 Hub Airports by Route Activity')
fig.update_layout(xaxis_title='Airport', yaxis_title='Total Routes')
fig.show()

con.close()


Top countries by airport count
          country  airport_count
0   United States           1512
1          Canada            430
2       Australia            334
3          Russia            264
4          Brazil            264
5         Germany            249
6           China            241
7          France            217
8  United Kingdom            167
9           India            148

Top airlines by route count
  airline  route_count
0      FR         2484
1      AA         2354
2      UA         2180
3      DL         1981
4      US         1960
5      CZ         1454
6      MU         1263
7      CA         1260
8      WN         1146
9      U2         1130

Top hub airports by total route activity
  airport_id                                              name  \
0       3682  Hartsfield Jackson Atlanta International Airport   
1       3830              Chicago O'Hare International Airport   
2       3364             Beijing Capital International Airport   
3        507      

Key insights:
- Large airport counts in a country often point to major travel markets or strong domestic aviation infrastructure.
- High airline route counts signal carriers with broad network reach and route diversification.
- The busiest hubs represent critical nodes in the network where route connectivity is concentrated.